# Exercise 01: Learning from Data – Loss, Train/Test, Cross-Validation, Bias–Variance

**Datasets:**
- **Bike counts in Zurich 2024–2025:** daily number of bikes at 16 automatic counting stations of the city of Zurich,
  with weather and calendar information (Open Data Stadt Zürich). File: `../Data/zurich_bike_counts_2024_2025.csv`.
- **Rental apartments in the canton of Zurich** (ZHAW course data) – the running example of the slides.
  File: `../Data/apartments_data_enriched_cleaned.csv`.

**Business question:** The city wants to estimate the number of cyclists on a working day from the time of year. How
complex should such a model be so that it also works on **new** days?

**Slides:** Part 02 «Learning from data» (loss functions, train/test split, k-fold cross-validation, under-/overfitting,
bias–variance trade-off).

**After this exercise you can ...**
- compute and compare squared and absolute loss by hand,
- explain why the training error is optimistically biased,
- use a hold-out split and k-fold cross-validation to choose model complexity,
- recognise under- and overfitting and relate them to bias and variance.

## Concept

**Loss and training objective.** A loss measures the error of *one* prediction. Training minimises the average loss
over the $n$ training examples:

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2, \qquad
\text{RMSE} = \sqrt{\text{MSE}}, \qquad
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|$$

Squared loss punishes large errors heavily (outlier-sensitive); absolute loss grows linearly (more robust).

**Generalisation error.** What we care about is the error on **new, unseen** data. It cannot be computed directly, so we
estimate it on data that were *not* used for fitting:

| approach | idea |
|---|---|
| hold-out split | fit on the training set (e.g. 80 %), evaluate once on the test set (20 %) |
| k-fold cross-validation | split the training set into $k$ folds; fit on $k-1$ folds, validate on the remaining one; repeat $k$ times; CV error $=\frac{1}{k}\sum_j \text{MSE}_j$ |

The test set is touched **only once** at the very end – hyperparameters (e.g. the polynomial degree) are chosen with CV.

**Under- and overfitting / bias–variance.** For squared-error loss the expected test error decomposes into

$$E\big[(y-\hat f(x))^2\big] = \text{Bias}^2 + \text{Variance} + \sigma^2_{\text{noise}}$$

- *Underfitting* (high bias): model too simple → high training **and** test error.
- *Overfitting* (high variance): model too flexible → low training error, high test error; the fit changes a lot with
  another training sample.
- The goal is to minimise the **sum**, not either term alone.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Loss functions by hand

Two models predict the number of bikes at a counting station on five days:

| day | observed $y$ | model A $\hat y$ | model B $\hat y$ |
|---|---|---|---|
| 1 | 1200 | 1500 | 1250 |
| 2 | 2500 | 2300 | 2550 |
| 3 | 3100 | 3300 | 3050 |
| 4 | 4000 | 3700 | 4050 |
| 5 | 4600 | 4500 | 3800 |

a) Compute the residuals, the **MSE**, the **RMSE** and the **MAE** for both models **by hand** (paper or calculator).

b) Which model is better according to RMSE, and which according to MAE? Explain the difference.

c) Verify your results with `numpy`.

In [ ]:
y       = np.array([1200, 2500, 3100, 4000, 4600])
y_hat_a = np.array([1500, 2300, 3300, 3700, 4500])
y_hat_b = np.array([1250, 2550, 3050, 4050, 3800])

# Compute residuals, MSE, RMSE and MAE for both models
# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the bike counts

a) Import the bike counts. Keep only **working days** (`weekend == 0` and `holiday == 0`) – otherwise the large
difference between working days and weekends dominates everything. Create the column `day_of_year`
(1 = 1 January, …, 366 = 31 December) from the column `date`.

b) Create a scatter plot of `bikes_total` (sum of all 16 counting stations) against `day_of_year`. Is the relationship
linear?

In [ ]:
# a) Import the data
bikes = pd.read_csv('../Data/zurich_bike_counts_2024_2025.csv', sep=',')
# Your code here

# b) Scatter plot bikes_total vs. day_of_year
# Your code here

**Your answer:** ...

## Exercise 3: Hold-out split and polynomial models

We describe the number of bikes as a polynomial of the day of the year

$$\hat y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d .$$

This is still a *linear* regression (linear in the parameters $\beta$). The **degree $d$** controls the model complexity.
The helper function `make_poly_model(degree)` below builds such a model (the features are standardised for numerical
stability).

a) Split the data into a training set (80 %) and a test set (20 %) with `random_state=42`.

b) Fit a polynomial of degree 1 (= simple linear regression) on the training set and compute the RMSE on the training
and on the test set.

In [ ]:
def make_poly_model(degree):
    """Polynomial regression model of the given degree."""
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(),
                         LinearRegression())

In [ ]:
X = df[['day_of_year']]
y = df['bikes_total']

# a) Train/test split (80/20, random_state=42)
X_train, X_test, y_train, y_test = ___

# b) Fit a polynomial of degree 1 and compute train and test RMSE
# Your code here

## Exercise 4: Under- and overfitting with a small training sample

Overfitting is most visible when there are only few training examples. Imagine the city had counted on only **30 days**.

a) Draw a random sample of 30 days from the training set (`X_train.sample(30, random_state=13)`, same rows for
`y_train`).

b) For each degree $d = 1, 2, \dots, 15$ fit `make_poly_model(d)` on the 30 days and compute the RMSE on these 30
training days and on the (full) test set. Plot both RMSE curves against the degree (use a logarithmic y-axis).

c) Plot the fitted curves for the degrees 1, 3 and 12 together with the 30 training days.

d) Which degree underfits, which overfits, which fits well? Why does the training error keep decreasing?

In [ ]:
# a) Sample of 30 training days
X_small = X_train.sample(30, random_state=13)
y_small = y_train.loc[X_small.index]

# b) Train and test RMSE for degree 1..15
degrees = range(1, 16)
rmse_train, rmse_test = [], []
for d in degrees:
    # Your code here
    pass

# Plot (use plt.yscale('log'))
# Your code here

In [ ]:
# c) Fitted curves for degree 1, 3 and 12
x_grid = pd.DataFrame({'day_of_year': np.linspace(X_small['day_of_year'].min(), X_small['day_of_year'].max(), 300)})

plt.figure(figsize=(8, 4))
plt.scatter(X_small['day_of_year'], y_small, color='black', s=20, label='30 training days')
for d in [1, 3, 12]:
    # Fit the model and plot its predictions on x_grid
    # Your code here
    pass
plt.ylim(0, 70000)
plt.xlabel('Day of the year')
plt.ylabel('Bikes per day')
plt.legend()
plt.show()

**Your answer:** ...

## Exercise 5: Choose the degree with 5-fold cross-validation

In practice we must not look at the test set to choose the degree. Instead we use k-fold cross-validation **on the
training data only**.

a) For $d = 1, \dots, 10$ compute the 5-fold CV RMSE on the **30 training days**
(use `cross_val_score(..., scoring='neg_root_mean_squared_error', cv=KFold(5, shuffle=True, random_state=42))`;
note the minus sign). Which degree does CV select?

b) Repeat a) with the **full training set**. Which degree is selected now and how large is the difference between
degree 3 and degree 10?

c) Fit the selected model on the full training set and evaluate it **once** on the test set.

d) Why does overfitting become much less severe with more data?

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(degree, X, y):
    """Mean 5-fold CV RMSE of a polynomial model."""
    # Your code here
    pass

# a) 30 training days
# Your code here

# b) Full training set
# Your code here

In [ ]:
# c) Fit the selected model on the full training set, evaluate once on the test set
# Your code here

**Your answer:** ...

## Exercise 6: Visualise bias and variance

Bias and variance describe what happens if we **repeat** the whole training on *other* samples of the same size.

a) Draw 50 different random samples of 30 days from the training set (`random_state=0, 1, ..., 49`). For each sample
fit a polynomial of degree 1, 3 and 12. Plot the 50 fitted curves per degree in three subplots
(use `ylim(0, 70000)`).

b) For each degree, compute the mean and the standard deviation of the 50 predictions for **day 180** (end of June).
What do you observe? Relate your observations to bias and variance.

In [ ]:
x_grid = pd.DataFrame({'day_of_year': np.linspace(1, 366, 300)})
x_180  = pd.DataFrame({'day_of_year': [180]})

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
pred_180 = {1: [], 3: [], 12: []}

for ax, d in zip(axes, [1, 3, 12]):
    for seed in range(50):
        # Draw a sample of 30 training days, fit the model, plot the curve on x_grid
        # and store the prediction for day 180 in pred_180[d]
        # Your code here
        pass
    ax.set_title(f'degree {d}')
    ax.set_ylim(0, 70000)
plt.show()

# b) Mean and standard deviation of the predictions for day 180
# Your code here

**Your answer:** ...

## Exercise 7: The slide example – 18 flats

The slides illustrate bias and variance with rents: *"predicting every rent by the overall mean → high bias; a degree-15
polynomial through 18 flats → high variance."* We now check this statement with real rental apartments from the canton
of Zurich (ZHAW course data: rent `price` in CHF/month and living area `area` in m²).

a) Import the apartment data (`sep=';'`), keep `area` and `price`, remove duplicates and missing values and keep only
flats with a rent between 1'000 and 5'000 CHF. Split the data 80/20 (`random_state=42`) and draw a sample of
**18 flats** from the training set (`random_state=0`).

b) Fit three models on the 18 flats: (1) the overall mean of the 18 rents, (2) a straight line (`make_poly_model(1)`),
(3) a polynomial of degree 15 (`make_poly_model(15)`). Compute the RMSE on the 18 flats and on the test set.

c) Plot the 18 flats and the three fitted curves for living areas from 20 to 200 m² (use `plt.ylim(0, 6000)`).

d) Which model has high bias, which one high variance? Explain the test error of the degree-15 polynomial.

In [ ]:
# a) Import the apartment data and draw 18 flats
apt = pd.read_csv('../Data/apartments_data_enriched_cleaned.csv', sep=';')
# Your code here

# b) Mean model, straight line and degree-15 polynomial
# Your code here

# c) Plot the three fitted curves
area_grid = pd.DataFrame({'area': np.linspace(20, 200, 400)})
# Your code here

**Your answer:** ...

## Exercise 8: Reflection

a) A colleague chooses the polynomial degree by trying all degrees and picking the one with the smallest **test** error.
What is the problem?

b) Why is k-fold cross-validation usually preferred over a single validation split when the data set is small?

c) Name two ways to reduce the variance of a model and one way to reduce its bias.

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')